In [1]:
from embedder import Embedder

embed = Embedder()

q1 = "How does approximate nearest neighbor search work?"

q1_embedded = embed.encode(q1)

2026-06-29 17:30:54.573677540 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [ ]:
print(f"""
        Q-1: The embedder returns a vector of {len(q1_embedded)} numbers.
        A-1: What's the first value (v[0])? == {q1_embedded[0].round(2)}
        """)


        Q-1: The embedder returns a vector of 384 numbers. 

        A-1: What's the first value (v[0])? == -0.02
        


In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [4]:
for doc in documents:
    if doc.get('filename')=="02-vector-search/lessons/07-sqlitesearch-vector.md":
        break

target_embedded_doc = embed.encode(doc.get('content'))

In [67]:
cosine_sim = target_embedded_doc.dot(q1_embedded)
print(f"""
      Q-2: What's the cosine similarity between the query and the document?
      A-2: {cosine_sim.round(2)}
        """)


      Q-2: What's the cosine similarity between the query and the document?
      A-2: 0.36
        


In [24]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
chunks_content_lst = [chunk.get('content') for chunk in chunks]

In [ ]:
import numpy as np
embeddings = embed.encode_batch(chunks_content_lst)
X = np.array(embeddings)
scores = X.dot(q1_embedded)
idx = np.argmax(scores)
idx, scores[idx]
print(f"""
    Q-3: Which chunk has the highest cosine similarity with the query?
    A-3: {chunks[idx].get('filename')}
    """)


    Q-3: Which chunk has the highest cosine similarity with the query?
    A-3: 02-vector-search/lessons/07-sqlitesearch-vector.md
    


In [33]:
from minsearch import VectorSearch
vindex = VectorSearch()
vindex.fit(X, chunks)

In [34]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)
print(f"""
      Q-4: Which file is the filename of the first result?
      A-4: {results[0].get('filename')}
      """)


      Q-4: Which file is the filename of the first result?
      A-4: 04-evaluation/lessons/05-search-metrics.md
      


In [64]:
query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)

vector_results = vindex.search(query_vector)
top_matching_vector = set([result.get('filename') for result in vector_results[:5]])
print(f'Vector Search Results: {top_matching_vector}')


from minsearch import Index
index = Index(text_fields=["content"])
index.fit(chunks)
index_results = index.search(query)
top_index_match = set([result.get('filename') for result in index_results[:5]])
print(f'Index Search Results: {top_index_match}')

Vector Search Results: {'02-vector-search/lessons/08-pgvector.md', '03-orchestration/lessons/05-rag.md'}
Index Search Results: {'02-vector-search/lessons/01-intro.md', '02-vector-search/lessons/02-embeddings.md', '03-orchestration/lessons/05-rag.md'}


In [61]:
print(f"""
      Q-5: Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?
      A-5: {list(top_matching_vector - top_index_match)[0]}""")


      Q-5: Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?
      A-5: 02-vector-search/lessons/08-pgvector.md


In [54]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [57]:
query = "How do I give the model access to tools?"
query_vector = embed.encode(query)
vector_results = vindex.search(query_vector)

from minsearch import Index
index = Index(text_fields=["content"],)
index.fit(chunks)
index_results = index.search(query)

In [60]:
print(f"""
      Q-6: Which file is ranked first after RRF?
      A-6: {rrf([vector_results,index_results])[0].get('filename')} """)



      Q-6: Which file is ranked first after RRF?
      A-6: 01-agentic-rag/lessons/13-function-calling.md 
